# 모듈 3. Tool Guardrails - 권한 기반 실행

기존 강의자료 `1. langchain/6. Tools_Agents/1. Tools_Agents.ipynb`, `2. Python Tools.ipynb`의 `@tool` 작성 방식을 그대로 사용하고, Tool 실행 직전에 권한 가드레일을 추가합니다.

## A/B 테스트 구조

- **A: 기존 코드 그대로 사용** - Agent가 선택한 Tool을 바로 실행합니다.
- **B: 기존 코드에 가드레일 추가** - Tool metadata를 기준으로 역할과 승인 필요 여부를 검사합니다.

In [ ]:
from langchain_core.tools import tool


@tool
def search_tool(query: str) -> str:
    """공개 웹 검색을 수행한다."""
    return f"검색 결과 3건: {query}"


@tool
def sql_read_tool(query: str) -> str:
    """분석용 SELECT 쿼리를 실행한다."""
    return f"분석용 SELECT 결과: {query}"


@tool
def send_email_tool(message: str) -> str:
    """관리자에게 이메일을 발송한다."""
    return f"이메일 발송 완료: {message}"


@tool
def delete_user_tool(user_id: str) -> str:
    """사용자 계정을 삭제한다."""
    return f"사용자 {user_id} 삭제 완료"


tools = {
    "search": search_tool,
    "sql_read": sql_read_tool,
    "send_email": send_email_tool,
    "delete_user": delete_user_tool,
}

tool_argument_keys = {
    "search": "query",
    "sql_read": "query",
    "send_email": "message",
    "delete_user": "user_id",
}


def select_tool(user_message: str) -> tuple[str, str]:
    if "삭제" in user_message:
        return "delete_user", "user-123"
    if "메일" in user_message:
        return "send_email", "관리자에게 장애 보고"
    if "매출" in user_message:
        return "sql_read", "SELECT * FROM sales"
    return "search", user_message


def invoke_tool(tool_name: str, argument: str) -> str:
    tool_obj = tools[tool_name]
    key = tool_argument_keys[tool_name]
    return tool_obj.invoke({key: argument})

## 1단계: A - 기존 Tool 호출 흐름 그대로 사용

기존 Tool/Agent 예제처럼 Agent가 Tool을 선택하면 바로 실행합니다. 이 경우 일반 사용자 요청에서도 삭제 Tool이 실행될 수 있습니다.

In [ ]:
user_role = "user"
user_message = "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."

tool_name, argument = select_tool(user_message)

print("사용자 역할:", user_role)
print("선택된 Tool:", tool_name)
print(invoke_tool(tool_name, argument))

### A안 문제점 확인

A안에서 보안 문제가 실제로 발생했는지 확인합니다.

In [ ]:
is_delete_tool = tool_name == "delete_user"
executed_without_role_check = is_delete_tool and user_role == "user"

print("A안 보안 진단")
print("- 사용자 역할:", user_role)
print("- 선택된 Tool:", tool_name)
print("- 삭제/쓰기 계열 Tool:", is_delete_tool)
print("- 권한 검사 없이 실행됨:", executed_without_role_check)
if executed_without_role_check:
    print("=> 문제: 일반 사용자가 계정 삭제 Tool을 실행했습니다.")

## 2단계: B - 기존 Tool에 권한 Guardrail 추가

Tool 정의는 그대로 두고, 실행 직전에 정책 테이블을 확인합니다. 프롬프트가 아니라 서버 코드에서 권한을 강제하는 것이 포인트입니다.

In [ ]:
role_rank = {
    "guest": 0,
    "user": 1,
    "analyst": 2,
    "manager": 3,
    "admin": 4,
}

tool_policies = {
    "search": {"risk": "safe", "required_role": "guest", "approval": False},
    "sql_read": {"risk": "read", "required_role": "analyst", "approval": False},
    "send_email": {"risk": "external_write", "required_role": "manager", "approval": True},
    "delete_user": {"risk": "delete", "required_role": "admin", "approval": True},
}


def guarded_invoke_tool(user_role: str, tool_name: str, argument: str) -> str:
    policy = tool_policies[tool_name]

    if role_rank[user_role] < role_rank[policy["required_role"]]:
        return (
            f"거부: {tool_name} Tool은 {policy['required_role']} 이상만 사용 가능합니다. "
            f"현재 역할={user_role}"
        )

    if policy["approval"]:
        return f"승인 대기: {tool_name} Tool은 실행 전 사람의 승인이 필요합니다."

    return invoke_tool(tool_name, argument)


test_cases = [
    ("user", "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."),
    ("analyst", "이번 달 매출 데이터를 조회해줘."),
    ("manager", "관리자에게 장애 보고 메일을 보내줘."),
    ("admin", "user-123을 삭제해줘."),
]

for role, message in test_cases:
    tool_name, argument = select_tool(message)
    print(f"[{role}] {message}")
    print(guarded_invoke_tool(role, tool_name, argument))
    print("-" * 70)

### B안 개선 확인

B안에서 기존 코드 흐름은 유지하면서 보안 문제가 줄어드는지 확인합니다.

In [ ]:
dangerous_tool_name, dangerous_argument = select_tool("일반 사용자지만 user-123을 삭제해줘.")
guarded_result = guarded_invoke_tool("user", dangerous_tool_name, dangerous_argument)

print("B안 보안 진단")
print("- 요청 Tool:", dangerous_tool_name)
print("- Guardrail 결과:", guarded_result)
print("- 위험 Tool 차단:", guarded_result.startswith("거부"))
print("=> 개선: Agent가 위험 Tool을 선택해도 서버 측 권한 검사에서 실행이 중단됩니다.")

## 정리

기존 Tool 작성 방식은 유지했습니다. 달라진 점은 Tool을 실행하기 직전에 역할, 위험도, 승인 필요 여부를 검사한다는 것입니다. Tool Guardrail은 Agent가 어떤 Tool을 골랐는지와 서버가 그 Tool 실행을 허용하는지를 분리합니다.